## **1. Raw Data Loading & Quality Diagnostics**

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd().parent / "data" / "raw" / "oulad"
print(DATA_DIR)
print(list(DATA_DIR.iterdir()))

In [ ]:
info = pd.read_csv(DATA_DIR / "studentInfo.csv")
reg = pd.read_csv(DATA_DIR / "studentRegistration.csv")
vle = pd.read_csv(DATA_DIR / "vle.csv")
courses = pd.read_csv(DATA_DIR / "courses.csv")

for name, df in [("studentInfo", info), ("studentRegistration", reg),
                  ("vle", vle), ("courses", courses)]:
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print(df.dtypes)
    print(df.head(3))

In [ ]:
svle = pd.read_csv(
    DATA_DIR / "studentVle.csv",
    usecols=["code_module", "code_presentation", "id_student", "id_site", "date", "sum_click"]
)
print(svle.shape)
print(svle.dtypes)
print(svle.head(10))

In [ ]:
print(svle["date"].describe())
sample_id = svle["id_student"].iloc[0]
one_student = svle[svle["id_student"] == sample_id].sort_values("date")
print(one_student.head(20))
print("\nSố dòng trùng cùng 1 (id_student, date, id_site):")
print(one_student.duplicated(subset=["date", "id_site"]).sum())

In [ ]:
full_dup = one_student.duplicated(
    subset=["code_module", "code_presentation", "id_student", "id_site", "date"]
)
print("Trùng toàn bộ key (module, presentation, student, site, date):", full_dup.sum())

dup_group = one_student[one_student["id_site"] == 546652]
print(dup_group.sort_values("date"))

In [ ]:
svle_clean = (
    svle.groupby(["code_module", "code_presentation", "id_student", "id_site", "date"], as_index=False)
        ["sum_click"].sum()
)
print("Trước gộp:", svle.shape, " Sau gộp:", svle_clean.shape)
print(f"Số dòng bị gộp: {svle.shape[0] - svle_clean.shape[0]}")

In [ ]:
print("Số loại activity_type:", vle["activity_type"].nunique())
print(vle["activity_type"].value_counts())

In [ ]:
print(info["final_result"].value_counts())
print(info["final_result"].value_counts(normalize=True).round(3))

In [ ]:
print(reg[["date_registration", "date_unregistration"]].describe())
n_withdrawn = reg["date_unregistration"].notna().sum()
print(f"Có date_unregistration: {n_withdrawn}/{len(reg)}")

In [ ]:
one_student_clean = svle_clean[svle_clean["id_student"] == sample_id].sort_values("date")
sample = one_student_clean.merge(vle[["id_site", "activity_type"]], on="id_site", how="left")
sample["week"] = sample["date"] // 7
weekly = sample.groupby(["week", "activity_type"])["sum_click"].sum().unstack(fill_value=0)
print(weekly)

per_day_variety = sample.groupby("date")["activity_type"].nunique()
print("\nSố loại activity_type khác nhau mỗi ngày (thống kê):")
print(per_day_variety.describe())

## **2. Visual EDA**

1. How imbalanced is the dropout label?
2. Which activity types dominate student interaction?
3. When do students interact most, and when do they withdraw?
4. Does a single day contain multiple activity types (justifies day-level transitions)?
5. What does one student's weekly behavior look like as a heatmap?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
FIG_DIR = Path.cwd().parent / "results" / "figures" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)